# GPT-2 Mini → Medical Chatbot with LoRA

**A beginner-friendly TensorFlow/Keras Colab lab.** This notebook reconstructs the **same 8-layer GPT architecture** from `GPT2_Mini_Chat.ipynb`, loads your **existing SODA-trained weights**, freezes them, and trains **only Q/V LoRA adapters** on medical dialogue examples.

### Learning goals
1. Compare the chatbot **before and after** medical domain adaptation.
2. See **trainable parameters versus frozen parameters**.
3. Train small LoRA matrices, save them separately, and toggle **adapter ON/OFF**.
4. Understand why **lower loss does not establish medical reliability**.

**Teaching/research use only. Not for diagnosis, treatment, triage or patient care.** The data can contain inaccurate or outdated responses. Use fictional test prompts, never patient-identifying information.

**Run:** Colab → Runtime → Change runtime type → GPU (T4 or better), then run all cells. The original weights must already exist in Google Drive; **this notebook does not retrain the base GPT**.

In [ ]:
# 1. Install dependencies and use the original architecture settings
!pip -q install tiktoken datasets

import os, random, math, json
import numpy as np
import tensorflow as tf
import keras
from keras import ops
import tiktoken
import matplotlib.pyplot as plt

# Keep float32 initially for easy debugging on Colab T4.
keras.mixed_precision.set_global_policy("float32")
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

N_LAYER = 8
N_HEAD = 8
N_EMBD = 512
BLOCK_SIZE = 256
DROPOUT = 0.0
enc = tiktoken.get_encoding("gpt2")
VOCAB_SIZE = enc.n_vocab

# Change these settings to experiment.
RANK = 8
ALPHA = 16
TRAIN_SAMPLES = 3000
VAL_SAMPLES = 200
EPOCHS = 2
BATCH_SIZE = 4
LEARNING_RATE = 1e-3
MAX_ANSWER_TOKENS = 90

BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/Teaching/Trained_Models/gpt_chat.weights.h5"
ADAPTER_PATH = "/content/drive/MyDrive/Colab Notebooks/Teaching/Trained_Models/gpt_chat_medical_lora.npz"
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 2. RECREATE YOUR ORIGINAL GPT MODEL (same source architecture)
class Block(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.ln1 = keras.layers.LayerNormalization(epsilon=1e-5)
        self.attn = keras.layers.MultiHeadAttention(N_HEAD, key_dim=N_EMBD // N_HEAD, dropout=DROPOUT)
        self.ln2 = keras.layers.LayerNormalization(epsilon=1e-5)
        self.mlp = keras.Sequential([
            keras.layers.Dense(4 * N_EMBD, activation="gelu"),
            keras.layers.Dense(N_EMBD),
            keras.layers.Dropout(DROPOUT),
        ])

    def call(self, x, training=False):
        h = self.ln1(x)
        x = x + self.attn(h, h, use_causal_mask=True, training=training)
        x = x + self.mlp(self.ln2(x), training=training)
        return x

class GPT(keras.Model):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.tok_emb = keras.layers.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_emb = keras.layers.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop = keras.layers.Dropout(DROPOUT)
        self.blocks = [Block() for _ in range(N_LAYER)]
        self.ln_f = keras.layers.LayerNormalization(epsilon=1e-5)

    def call(self, idx, training=False):
        T = ops.shape(idx)[1]
        x = self.tok_emb(idx) + self.pos_emb(ops.arange(T))
        x = self.drop(x, training=training)
        for block in self.blocks:
            x = block(x, training=training)
        x = self.ln_f(x)
        logits = ops.matmul(x, ops.transpose(ops.cast(self.tok_emb.embeddings, x.dtype)))
        return ops.cast(logits, "float32")

model = GPT()
_ = model(np.zeros((1, BLOCK_SIZE), dtype=np.int32))
BASE_PARAMS = model.count_params()
print(f"Original GPT: {BASE_PARAMS:,} parameters")

In [ ]:
# 3. Load your trained weights. No SODA pretraining happens here.
from google.colab import drive
drive.mount("/content/drive")
assert os.path.exists(BASE_PATH), f"Missing original weights: {BASE_PATH}. Set BASE_PATH to your file."
model.load_weights(BASE_PATH)
print("Loaded the ORIGINAL trained GPT weights.")

def generate(prompt, max_new_tokens=70, top_k=20, temperature=0.7, use_adapter=True):
    """Generate just one Bot turn; LoRA can be enabled/disabled."""
    ids = enc.encode_ordinary(prompt)
    new = []
    for _ in range(max_new_tokens):
        logits = model(np.array([ids[-BLOCK_SIZE:]], dtype=np.int32), training=False)[0, -1]
        logits = logits / max(temperature, 0.05)
        top_scores, top_ids = tf.math.top_k(logits, k=top_k)
        chosen = int(top_ids[int(tf.random.categorical(top_scores[None, :], 1)[0, 0])])
        if chosen == enc.eot_token or "\n" in enc.decode([chosen]):
            break
        ids.append(chosen); new.append(chosen)
    return enc.decode(new).strip()

QUESTIONS = [
    "I have had a headache since yesterday. What should I do?",
    "I have a sore throat and a mild cough. What should I consider?",
    "What does a balanced diet mean?",
]
base_answers = []
for q in QUESTIONS:
    a = generate(f"User: {q}\nBot:")
    base_answers.append(a)
    print("Patient:", q, "\nOriginal GPT:", a, "\n")

## 4. Attach LoRA only to Q and V

For each transformer attention block, the original projection remains **frozen**:

\[Q = XW_Q + (\alpha/r)(XA_Q)B_Q,\quad V=XW_V+(\alpha/r)(XA_V)B_V\]

In Keras, `MultiHeadAttention` stores Q/V kernels as **three-dimensional** tensors `(512, 8, 64)`. The small adapter below performs the same standard causal attention operation, with added trainable low-rank Q/V updates. Its frozen base is the **original loaded** attention layer, kept in an unmodified pretrained GPT object.

**Important:** We test that turning the adapter off produces the same output as the original Keras attention, before training. The adapter is a teaching implementation, not a generic production LoRA library.

In [ ]:
# 4. LoRA attention wrapper: 4 trainable matrices per block; base MHA stays frozen.
class LoRAAttention(keras.layers.Layer):
    def __init__(self, original_attention, rank=8, alpha=16, **kwargs):
        super().__init__(**kwargs)
        self.base = original_attention
        self.base.trainable = False
        self.rank = rank
        self.scale = alpha / rank
        self.enabled = True

    def build(self, input_shape):
        d = N_EMBD; h = N_HEAD; k = d // h
        # B starts at zero: adding LoRA initially changes nothing.
        self.Aq = self.add_weight(name="Aq", shape=(d, self.rank), initializer=keras.initializers.RandomNormal(stddev=0.01), trainable=True)
        self.Bq = self.add_weight(name="Bq", shape=(self.rank, h, k), initializer="zeros", trainable=True)
        self.Av = self.add_weight(name="Av", shape=(d, self.rank), initializer=keras.initializers.RandomNormal(stddev=0.01), trainable=True)
        self.Bv = self.add_weight(name="Bv", shape=(self.rank, h, k), initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, x, training=False):
        # The same learned Q/K/V projections stored in your trained MultiHeadAttention.
        qd, kd, vd, od = (self.base._query_dense, self.base._key_dense,
                           self.base._value_dense, self.base._output_dense)
        q = tf.einsum("btd,dhk->bthk", x, qd.kernel) + qd.bias
        k = tf.einsum("btd,dhk->bthk", x, kd.kernel) + kd.bias
        v = tf.einsum("btd,dhk->bthk", x, vd.kernel) + vd.bias
        if self.enabled:
            q += self.scale * tf.einsum("btr,rhk->bthk", tf.matmul(x, self.Aq), self.Bq)
            v += self.scale * tf.einsum("btr,rhk->bthk", tf.matmul(x, self.Av), self.Bv)
        # Standard causal multi-head attention (dropout=0 in the original model).
        scores = tf.einsum("bthk,bshk->bhts", q, k) / math.sqrt(N_EMBD // N_HEAD)
        T = tf.shape(x)[1]
        allowed = tf.linalg.band_part(tf.ones((T, T), dtype=tf.bool), -1, 0)
        scores = tf.where(allowed[None, None, :, :], scores, tf.constant(-1e9, scores.dtype))
        weights = tf.nn.softmax(scores, axis=-1)
        mixed = tf.einsum("bhts,bshk->bthk", weights, v)
        return tf.einsum("bthk,hkd->btd", mixed, od.kernel) + od.bias

# Build adapters OUTSIDE the already-built base GPT, avoiding Keras tracking errors.
original_attentions = [b.attn for b in model.blocks]
xcheck = tf.random.normal([1, 12, N_EMBD])
test_adapter = LoRAAttention(original_attentions[0], RANK, ALPHA)
_ = test_adapter(xcheck)
error = float(tf.reduce_max(tf.abs(original_attentions[0](xcheck, xcheck, use_causal_mask=True, training=False) - test_adapter(xcheck))))
print("Maximum MHA vs adapter difference (before training):", error)
assert error < 0.001, "Attention mismatch: check TensorFlow/Keras version before training."

class MedicalLoRAGPT(keras.Model):
    def __init__(self, pretrained, rank, alpha):
        super().__init__()
        self.base = pretrained
        self.base.trainable = False
        self.adapters = [LoRAAttention(b.attn, rank, alpha) for b in pretrained.blocks]

    def call(self, idx, training=False):
        T = ops.shape(idx)[1]
        x = self.base.tok_emb(idx) + self.base.pos_emb(ops.arange(T))
        x = self.base.drop(x, training=False)
        for block, attn in zip(self.base.blocks, self.adapters):
            h = block.ln1(x)
            x = x + attn(h, training=training)
            x = x + block.mlp(block.ln2(x), training=False)
        x = self.base.ln_f(x)
        return ops.cast(ops.matmul(x, ops.transpose(ops.cast(self.base.tok_emb.embeddings, x.dtype))), "float32")

base_model = model
model = MedicalLoRAGPT(base_model, RANK, ALPHA)
_ = model(tf.zeros((1, 8), dtype=tf.int32))
adapters = model.adapters
train_vars = model.trainable_variables
expected = N_LAYER * 2 * (N_EMBD * RANK + RANK * N_EMBD)
actual = sum(int(np.prod(v.shape)) for v in train_vars)
print(f"LoRA trainable parameters: {actual:,} / base parameters: {BASE_PARAMS:,}")
print(f"Trainable percentage: {100*actual/BASE_PARAMS:.3f}%")
assert actual == expected, (actual, expected)
assert len(train_vars) == 4 * N_LAYER, "Only four A/B LoRA matrices per block should train."


In [ ]:
# 5. Load MedDialog (stream a small, reproducible teaching subset).
from datasets import load_dataset

# MedDialog dataset variants may change their field names over time.
stream = load_dataset("OpenMed/MedDialog", split="train", streaming=True)
stream = stream.shuffle(seed=SEED, buffer_size=2000)

def get_pair(row):
    # Fail clearly if the dataset schema changes.
    patient = row.get("patient_message")
    doctor = row.get("doctor_response")
    if patient is None or doctor is None:
        raise ValueError(f"Dataset schema changed. Columns: {list(row)}")
    return str(patient).strip(), str(doctor).strip()

# Limit both turns to the context window, and never truncate away the answer silently.
examples = []
for row in stream:
    patient, doctor = get_pair(row)
    if not patient or not doctor:
        continue
    prefix = f"User: {patient}\nBot:"
    answer = " " + doctor + "\n"
    prompt_ids = enc.encode_ordinary(prefix)
    answer_ids = enc.encode_ordinary(answer)
    if len(prompt_ids) < 8 or len(answer_ids) < 8:
        continue
    if len(prompt_ids) + len(answer_ids) > BLOCK_SIZE:
        continue
    examples.append((prompt_ids, answer_ids))
    if len(examples) >= TRAIN_SAMPLES + VAL_SAMPLES:
        break
assert len(examples) > VAL_SAMPLES + 100, "Too few short examples. Increase streamed rows or relax filtering."
train_ex = examples[:min(TRAIN_SAMPLES, len(examples)-VAL_SAMPLES)]
val_ex = examples[-VAL_SAMPLES:]
print(f"Training pairs: {len(train_ex)}, validation pairs: {len(val_ex)}")
print("Example question:", enc.decode(train_ex[0][0])[:180])
print("Example answer:", enc.decode(train_ex[0][1])[:180])

In [ ]:
# 6. Teacher-forcing data: compute loss ONLY on the doctor's answer tokens.
# Labels are -100 wherever the prompt/padding should not count in loss.
def make_arrays(pairs):
    xs = np.zeros((len(pairs), BLOCK_SIZE-1), np.int32)
    ys = np.zeros((len(pairs), BLOCK_SIZE-1), np.int32)
    masks = np.zeros((len(pairs), BLOCK_SIZE-1), np.float32)
    for i, (p, a) in enumerate(pairs):
        ids = p + a
        x, y = ids[:-1], ids[1:]
        xs[i, :len(x)] = x
        ys[i, :len(y)] = y
        # At position len(prompt)-1 we first predict the answer.
        masks[i, len(p)-1:len(y)] = 1.0
    return xs, ys, masks

xtr, ytr, mtr = make_arrays(train_ex)
xva, yva, mva = make_arrays(val_ex)
train_ds = tf.data.Dataset.from_tensor_slices((xtr, ytr, mtr)).shuffle(len(xtr), seed=SEED).batch(BATCH_SIZE).prefetch(2)
val_ds = tf.data.Dataset.from_tensor_slices((xva, yva, mva)).batch(BATCH_SIZE).prefetch(2)

# Safe sparse cross entropy averaged only over doctor-answer positions.
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction="none")
def answer_loss(x, y, mask, training=False):
    logits = model(x, training=training)
    token_loss = loss_fn(y, logits)
    return tf.reduce_sum(token_loss * mask) / tf.reduce_sum(mask)

optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
@tf.function
def train_step(x, y, mask):
    with tf.GradientTape() as tape:
        loss = answer_loss(x, y, mask, training=True)
    grads = tape.gradient(loss, train_vars)
    optimizer.apply_gradients(zip(grads, train_vars))
    return loss

history = {"train": [], "validation": []}
for epoch in range(EPOCHS):
    epoch_losses = []
    for step, batch in enumerate(train_ds):
        epoch_losses.append(float(train_step(*batch)))
        if (step+1) % 100 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS}, step {step+1}, answer loss {np.mean(epoch_losses[-100:]):.3f}")
    val_losses = [float(answer_loss(*batch, training=False)) for batch in val_ds]
    history["train"].append(float(np.mean(epoch_losses)))
    history["validation"].append(float(np.mean(val_losses)))
    print(f"Epoch {epoch+1}: train={history['train'][-1]:.3f}, validation={history['validation'][-1]:.3f}")

plt.plot(range(1, EPOCHS+1), history["train"], marker="o", label="Training")
plt.plot(range(1, EPOCHS+1), history["validation"], marker="o", label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Doctor-answer cross entropy"); plt.legend(); plt.show()

In [ ]:
# 7. Compare the exact same user prompts with LoRA ON versus OFF.
def ask(question, adapter_on=True, max_tokens=MAX_ANSWER_TOKENS):
    for a in adapters:
        a.enabled = adapter_on
    return generate(f"User: {question}\nBot:", max_new_tokens=max_tokens)

for q in QUESTIONS:
    print("PATIENT:", q)
    print("ORIGINAL GPT (LoRA OFF):", ask(q, False))
    print("MEDICAL LoRA (ON):    ", ask(q, True))
    print("-" * 72)

# Restores LoRA ON after comparison.
for a in adapters: a.enabled = True
print("Remember: coherent answers can still contain dangerous medical misinformation.")

In [ ]:
# 8. Save only the small adapters (NOT the pretrained model).
os.makedirs(os.path.dirname(ADAPTER_PATH), exist_ok=True)
weights = {f"block_{i}_{name}": np.array(getattr(a, name)) for i, a in enumerate(adapters)
           for name in ("Aq", "Bq", "Av", "Bv")}
np.savez_compressed(ADAPTER_PATH, **weights)
print("Saved LoRA adapter:", ADAPTER_PATH)
print("Adapter size (MB):", round(os.path.getsize(ADAPTER_PATH)/1e6, 2))

def load_adapter(path=ADAPTER_PATH):
    saved = np.load(path)
    for i, a in enumerate(adapters):
        for name in ("Aq", "Bq", "Av", "Bv"):
            getattr(a, name).assign(saved[f"block_{i}_{name}"])
    print("Loaded adapter from", path)

# OPTIONAL: load_adapter() in a later session after rebuilding GPT,
# loading the original BASE_PATH weights and attaching LoRA with the same rank.

## 9. Student experiments

1. Change `RANK` to **2, 4, 8, or 16**, restart the runtime, and compare parameter counts, validation loss and generated text.
2. Keep the base checkpoint unchanged, and compare the same fictional symptom questions with adapter **OFF vs ON**.
3. Hold out a small set of medically themed questions with new phrasing. Does the adaptation generalize?
4. Why might a lower answer-token loss **not** mean the medical advice is correct?

### Limits
- The pretrained model is small, not instruction-tuned, and may give unhelpful text even after LoRA.
- This is a **patient→doctor next-turn modeling** exercise, not clinical validation.
- Do not put real patient data into the public dataset or notebook.
- Train/validation rows from the same source may overlap semantically; do not claim clinical generalization.
- Only LoRA A/B matrices are trained; the original GPT checkpoint is never overwritten.